In [59]:
import json
import pandas as pd

In [60]:
df = pd.read_json("predicts/pool_1_prepared.json")

In [63]:
table_crits = """
| Критерий | ОПИСАНИЕ | ПРИМЕРЫ |
| :--- | :--- | :--- |
| **Из диалога понятно, что сотрудник пытается продать "Дебетовую карту"** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник сказал о дебетовой карте: "Предлагаю вам оформить дебетовую карту". Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь о дебетовой карте, то такая ситуация тоже подходит под критерий.<br>**Обратите внимание:** Сотрудник в диалоге с клиентом может использовать разное название продукта "Дебетовая карта". Может быть озвучено просто карта или Сберкарта/банковская карта, но по контексту диалога должно быть точно понятно что речь идет **о дебетовой карте**. Если из контекста непонятно о какой карте идет речь - не подходит. | - Сотрудник называет лимит на переводы внутри Сбера в 300000 в месяц;<br>- Сотрудник говорит, что плановый перевыпуск карты бесплатный;<br>- По дебетовой карте будет доступно 3 категории кешбэка на выбор. |
| **Сотрудник озвучил корректно финальную стоимость карты/Сберкарты** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник озвучил корректную финальную стоимость дебетовой карты: "стоимость карты 299 рублей" (на самом деле дебетовая карта бесплатная), далее сотрудник процессе диалога говорит клиенту, что карта бесплатная". Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь о корректной финальной стоимости обслуживания дебетовой карты, то такая ситуация тоже подходит под критерий. | - Сотрудник озвучивает, что дебетовая карта стоит 150 руб. (на самом деле дебетовая карта бесплатная), далее при оформлении карты сотрудник сообщает, что согласно новым условиям карта бесплатная (т.е. исправляется)<br>- Сотрудник озвучивает, что карта бесплатная (и на самом деле дебетовая карта бесплатная) |
| **Сотрудник сказал, что по карте/Сберкарте отсутствует комиссия при оплате услуг ЖКХ в удаленных каналах связи** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник сказал, что оплата ЖКХ в удаленных каналах обслуживания (связи) без комиссии : "если Вы будете оплачивать услуги ЖКХ в банкомате, то комиссии не будет". Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь идет об оплате ЖКХ без комиссии в удаленных каналах обслуживания (связи), то такая ситуация тоже подходит под критерий.<br>**Обратите внимание:** если сотрудник **не упоминает удаленные каналы обслуживания (связи)** и говорит об отсутствии комиссии при оплате ЖКХ, то под критерий это **не** подходит. | - Сотрудник говорит, что оплата ЖКХ по карте в банкомате будет бесплатной.<br>- Сотрудник говорит, что при оплате ЖКХ картой в устройстве самообслуживания комиссии не будет. |
| **Сотрудник сказал, о сумме лимита на переводы внутри Сбера в месяц** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник сказал о сумме лимита на переводы внутри Сбера в месяц: "По карте существует лимит на бесплатные переводы внутри Сбера в месяц в размере 300 тыс.руб".<br>**Обратите внимание:** если сотрудник озвучивает, только сумму лимита на переводы, без уточнения про переводы **внутри Сбера**, то такая ситуация **не** подходит под критерий. Аналогично если сотрудник **не** уточнил, что лимит именно **в месяц**.<br>**Обратите внимание:** в данном критерии сотрудник должен озвучить сумму лимита, а не просто что-либо сказать про лимит на переводы. Озвученную сумму лимита **не** проверяем на корректность. | - Сотрудник говорит, что по карте есть лимит по переводам в месяц внутри Сбера, это 300 тыс.руб.<br>- Сотрудник озвучивает, что с помощью данной карты вы можете бесплатно переводить клиентам до 400 тыс.руб. в месяц |
| **Сотрудник озвучил корректно финальную сумму лимита на переводы внутри Сбера в месяц** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник озвучил корректно финальную сумму лимита на переводы внутри Сбера в месяц: "По карте вы можете осуществлять переводы без комиссии до 400 тыс.руб. (на самом деле лимит по переводам внутри Сбера - 300тыс. руб. в месяц), извините, я ошибся лимит по переводам по картам сейчас составляет 300 тыс.руб." Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь идет о корректном финальном лимите на переводы в месяц, то такая ситуация тоже подходит под критерий.<br>**Обратите внимание:** если сотрудник озвучивает несколько сумм лимитов на переводы, то необходимо определить какая из сумм лимитов является финальной и если найденная сумма лимита совпадает с информацией из пункта 1, тогда мы считаем, что сотрудник корректно озвучил финальную сумму лимита на переводы и, следовательно, такая ситуация под критерий подходит.<br>**Обратите внимание:** если сотрудник озвучивает, только сумму лимита на переводы, без уточнения про переводы **внутри Сбера**, то такая ситуация **не** подходит под критерий, даже если сотрудник корректно озвучил финальную сумму лимита. Аналогично если сотрудник **не** уточнил, что лимит именно **в месяц**.<br>**Обратите внимание:** если сотрудник называет увеличенный лимит на переводы (500 тыс./1 млн.) и **не** называет стандартный лимит в 300 тыс., то это под критерий **не** подходит. Другими словами, при поиске финальной суммы лимита не нужно рассматривать повышенные лимиты на переводы от подписок СберПрайм/СберПрайм+. | - Сотрудник говорит, что по карте в месяц есть лимит по переводам внутри Сбера, это 500 тыс.руб. ( на самом деле: 300 тыс.руб), позже сотрудник в процессе беседы с клиентом исправляется и озвучивает 300 тыс.руб.<br>- Сотрудник говорит, что лимит по переводам 300 тыс.руб. в месяц внутри Сбера (и на самом деле: 300 тыс.руб) |
| **Сотрудник сказал, о количестве категорий кешбэка по карте/Сберкарте** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник сказал, о количестве кэшбэка по карте/Сберкарты: "к карте Вы можете подключить 3 категории кэшбэка по программе Спасибо". Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь идет о количестве кэшбэка по карте/Сберкарте, то такая ситуация тоже подходит под критерий.<br>**Обратите внимание:** количество категорий кэшбэка не проверяем на корректность. | - Сотрудник говорит, что по карте есть возможность копить бонусы по 3 выбранным кэшбэкам.<br>- Сотрудник озвучивает, что с помощью данной карты вы можете подключиться к программе Лояльности и выбрать 5 категории кэшбэка на свое усмотрение. |
| **Сотрудник озвучил информацию о сроке действия карты/Сберкарты** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник сказал, о сроке действия карты: "Срок действия Вы можете посмотреть на лицевой стороне карты под её номером". Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь идет о сроке действия карты, то такая ситуация тоже подходит под критерий.<br>**Обратите внимание:** Если сотрудник озвучивает месяц и год окончания срока действия карты или говорит, что срок действия указан на карте, то такая ситуация под критерий подходит. Месяц и год окончания срока действия карты не проверяем на корректность | - Сотрудник говорит клиенту, что срок действия карты можете посмотреть на лицевой стороне карты под её номером.<br>- Сотрудник говорит клиенту, что срок действия указан на карте<br>- Сотрудник говорит клиенту, что срок действия карты до 08.2035 |
| **Сотрудник озвучил информацию об автоматическом перевыпуске карты/Сберкарты** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник сказал, об автоматическом перевыпуске карты/Сберкарты: "если Вы не закроете счет карты, то она перевыпустится автоматически". Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь идет об автоматическом перевыпуске карты/Сберкарты, то такая ситуация тоже подходит под критерий. | - Сотрудник говорит клиенту, что карта перевыпускается автоматически, если ее счет не закрыт.<br>- Сотрудник говорит клиенту, если Вы не будете пользоваться картой и не закроете ее счет, то она автоматически перевыпустится. |
| **Сотрудник озвучил корректно финальную информацию о сроке действия карты/Сберкарты** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник сказал корректно финальную информацию о сроке действия карты/Сберкарты : "Карта бессрочная (на самом деле карта выпускается на 10 лет), Елена Андреевна, извините я ошибся, срок действия карты 10 лет". Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь идет о корректной финальной информации о сроке действия карты/Сберкарты, то такая ситуация тоже подходит под критерий. | - Сотрудник говорит клиенту, что карта выпускается на 1 год (на самом деле карта выпускается на 10 лет), в момент оформления карты сотрудник исправляется и озвучивает срок действия карты 10 лет.<br>- Сотрудник говорит клиенту, что карта выпускается на 10 лет. (На самом деле карта выпускается на 10 лет) |
| **Сотрудник сказал, если карта/СберКарта заблокирована/не используется клиентом необходимо закрыть ее счет** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник сказал, что при блокировке/неиспользовании карты/СберКарты необходимо закрыть ее счет : "если вы не пользуетесь картой/СберКартой, то необходимо закрыть ее счет ". Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь идет о закрытии счета карты/Сберкарты, то такая ситуация тоже подходит под критерий.<br>**Обратите внимание:** проверяется факт озвучивания сотрудником закрытие счета карты, способы закрытия счета карты не анализируются | - Сотрудник говорит клиенту, что если карта не нужна, то ее счет необходимо закрыть.<br>- Сотрудник говорит клиенту, что при блокировке карты, счет не закрывается, необходимо оформить заявление на закрытие счета в СберБанк Онлайн/офисе банка/ по номеру 900. |
| **Сотрудник озвучил корректно финальную стоимость перевыпуска карты/Сберкарты при плановом перевыпуске** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник корректно озвучил финальную стоимость планового перевыпуска/технической неисправности карты/Сберкарты : " перевыпуск карты стоит 150 руб. извините если перевыпуск плановый, то это совершенно бесплатно ". Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь идет о корректной финальной стоимости планового перевыпуска/технической неисправности карты/Сберкарты, то такая ситуация тоже подходит под критерий.<br>**Обратите внимание:** Если сотрудник озвучивает несколько стоимостей планового перевыпуска/технической неисправности карты/Сберкарты, то необходимо определить какая из стоимости является финальной и если найденная стоимость совпадает со стоимостью из пункта 1, тогда мы считаем, что сотрудник корректно озвучил финальную стоимость планового перевыпуска/технической неисправности карты/Сберкарты и, следовательно, такая ситуация под критерий подходит.<br>Сотрудник должен сказать про какой именно перевыпуск идет речь (плановый перевыпуск/техническая неисправность). Если сотрудник не озвучивает, то считаем, что озвучил некорректно. | - Сотрудник говорит клиенту, что перевыпуск карты стоит 150 руб. (а на самом деле бесплатный), дальше в процессе диалога с клиентом, сотрудник озвучивает, что плановый перевыпуск бесплатный (т.е. сотрудник исправляется)<br>- Сотрудник говорит клиенту, что перевыпуск будет бесплатным при тех.неисправности карты и ее плановом перевыпуске (и на самом деле перевыпуск будет бесплатным при тех.неисправности карты или ее плановом перевыпуске.) |
| **Сотрудник озвучил корректно финальную стоимость перевыпуска карты/Сберкарты при утрате/изменении личных данных** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник корректно озвучил финальную стоимость перевыпуска при утрате/изменении личных данных карты/Сберкарты : " перевыпуск карты стоит 299 руб. извините при утрате карты стоимость перевыпуска будет 150 руб.". Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь идет о корректной финальной стоимости перевыпуска при утрате/изменении личных данных карты/Сберкарты, то такая ситуация тоже подходит под критерий.<br>**Обратите внимание:** Если сотрудник озвучивает несколько стоимостей перевыпуска при утрате/изменении личных данных карты/Сберкарты, то необходимо определить какая из стоимости является финальной и если найденная стоимость совпадает со стоимостью из пункта 1, тогда мы считаем, что сотрудник корректно озвучил финальную стоимость перевыпуска при утрате/изменении личных данных карты/Сберкарты и, следовательно, такая ситуация под критерий подходит.<br>Сотрудник должен сказать про какой именно перевыпуск идет речь (при утрате/изменении личных данных). Если сотрудник не озвучивает, то считаем, что озвучил некорректно. | - Сотрудник говорит клиенту, что перевыпуск карты стоит 399 (а на самом деле 150 руб), дальше в процессе диалога с клиентом, сотрудник озвучивает, что при смене личных данных перевыпуск будет стоить 150 руб. (т.е. сотрудник исправляется)<br>- Сотрудник говорит клиенту, что стоимость перевыпуска карты/Сберкарты будет 150 руб. при утрате и изменении личных данных клиента (и на самом деле стоимость перевыпуска карты/Сберкарты будет 150 руб. при утрате и изменении личных данных клиента) |
| **Сотрудник сказал, что для оформления карты нужно обязательно оформить подписку Прайм/Прайм+** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник озвучил, что для оформления карты нужно обязательно оформить подписку Прайм/Прайм+: " при оформлении карты необходимо обязательно подключить Прайм/Прайм+ " (На самом деле подключение подписки Прайм/Прайм+ необязательно). Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь об обязательном подключении подписки Прайм/Прайм+, то такая ситуация тоже подходит под критерий. | - Сотрудник озвучивает, что для оформления карты нужно обязательно подключить Прайм/Прайм+<br>- Сотрудник озвучивает, что подписка Прайм/Прайм+ является обязательным условием при выпуске Сберкарты.<br>- Сотрудник озвучивает, что карта оформляется только с Прайм<br>- Сотрудник говорит, что согласно новым условиям карта выдается вместе с подпиской Прайм/Прайм+ |
| **Сотрудник сказал, что к новой Сберкарте открывается Платежный счет** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник озвучил при оформлении карты у клиента автоматически открывается платежный счет: "при оформлении карты автоматически открывается Платежный счет. " | - Сотрудник озвучивает, что при оформлении карты автоматически открывается Платежный счет.<br>- Сотрудник озвучивает, что теперь у карты есть Платежный счет |
| **Сотрудник озвучил возможность привязки карты к новому платежному счету** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник озвучил, что карту можно оформить с новым платежным счетом: "карту оформляем с новым платежным счетом ". Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь о новом Платежном счете , то такая ситуация тоже подходит под критерий. | - Сотрудник озвучивает, что при оформлении карты открывается новый Платежный счет.<br>- Сотрудник озвучивает, что к новой карте оформляется новый платежный счет<br>- Сотрудник озвучивает, что к карте можно привязать новый или старый платежный счет |
| **Сотрудник озвучил возможность привязки карты к старому платежному счету** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник озвучил, что карту можно привязать к старому платежному счету: "Новую карту можем привязать к старому платежному счету ". Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь о возможности привязать карту к старому платежному счету , то такая ситуация тоже подходит под критерий. | - Сотрудник озвучивает, что при оформленную карту мы можем привязать к старому платежному счету.<br>- Сотрудник озвучивает, что сейчас есть возможность новую карту подключить к старому платежному счету<br>- Сотрудник озвучивает, что к карте есть возможность привязать как новый платежный счет, так и старый |
| **Сотрудник озвучил клиенту, что необходимо проверить документы по операции** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник предложил проверить условия страхования, документы: "Ознакомьтесь с документами". Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь идет об ознакомлении с условиями и документами, то такая ситуация тоже подходит под критерий. | - Сотрудник говорит клиенту, что необходимо проверить документы.<br>- Сотрудник говорит клиенту, что необходимо проверить документы и условия по программе.<br>- Сотрудник говорит клиенту проверить на терминале условия и документы по операции. |
| **Сотрудник озвучил клиенту, что необходимо подтвердить операцию (поставить галочку/зеленая кнопка)** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник предложил клиенту подтвердить операцию: "Подтвердите операцию, нажмите зеленую кнопку". Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь идет о подтверждении операции, то такая ситуация тоже подходит под критерий. | - Сотрудник говорит клиенту, что необходимо подтвердить операцию, проставив галку<br>- Сотрудник говорит клиенту, что необходимо подтвердить операцию, нажав зеленую кнопку на терминале |
| **Сотрудник озвучил условия закрытия СберКарты** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник озвучил условия закрытия карты/СберКарты: "Карта закрывается в течение 30 дней в офисе банка". Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь идет о закрытии карты/Сберкарты, то такая ситуация тоже подходит под критерий.<br>**Обратите внимание:** к условиям закрытия карты относятся способы ее закрытия (СберБанк Онлайн, офис банка, номер 900) и срок закрытия в 30 дней. Сотрудник должен озвучить хотя бы один способ закрытия вместе с упоминанием срока закрытия. Срок закрытия в 30 дней на корректность проверять не нужно. | - Сотрудник говорит клиенту, что карту можно закрыть через СберБанк Онлайн<br>- Сотрудник говорит клиенту, что карта закрывается в течение 30 дней. Заявление на закрытие можно подать через берБанк Онлайн, офис банка, по номеру 900. |
| **Сотрудник озвучил корректно финальные условия закрытия карты/СберКарты** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник корректно озвучил финальные условия закрытия карты/СберКарты : "Карта закрывается у нас в офисе в течении 5 дней (на самом деле карта закрывается в течение 30 дней), а ойшибся, счет будет закрыт только через 30 дней, так как пробирается баланс карты". Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь идет о корректных финальных условиях закрытия карты/Сберкарты, то такая ситуация тоже подходит под критерий.<br>**Обратите внимание:** к условиям закрытия карты относятся способы ее закрытия (СберБанк Онлайн, офис банка, номер 900) и срок закрытия в 30 дней.<br>Если сотрудник озвучивает несколько условий закрытия карт, то необходимо определить какие из условий являются финальными и если найденные условия совпадает с информацией из пункта 1, тогда мы считаем, что сотрудник корректно озвучил финальные условия и, следовательно, такая ситуация под критерий подходит.<br>**Обратите внимание:** срок закрытия в 30 дней проверяем на корректность. | - Сотрудник говорит клиенту, что счёт карты можно не закрывать, если не будете пользоваться.(на самом деле способы закрытия карты-СберБанк Онлайн, офис банка, по номеру 900. Счет карты закрывается в течение 30 дней), далее сотрудник корректирует свои слова и говорит, что счет закрывается через 30 дней после проверки баланса. Закрыть карты можно в СберБанк Онлайн, офис банка, по номеру 900. (т.е. исправляется)<br>- Сотрудник говорит клиенту, что счёт карты закрывается в течение 30 дней, это можно сделать в СберБанк Онлайн, офис банка, по номеру 900. (и на самом деле карта закрывается в течение 30 дней через в СберБанк Онлайн, офис банка, по номеру 900.)<br>- Сотрудник говорит клиенту, что счёт карты закрывается в течение 30 дней, это можно сделать номеру 900. (и на самом деле карта закрывается в течение 30 дней через в СберБанк Онлайн/офис банка/ по номеру 900.) |
| **Сотрудник озвучил тариф дополнительной опции СберМобайл тарифа СберKIDS** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник озвучил тариф дополнительной опции СберМобайл тарифа СберKIDS: "тарифа СберKIDS- 300 минут на звонки по РФ, 30 ГБ, 30 СМС.".<br>**Обратите внимание:** сотрудники должен озвучить **все** основные опции по тарифу (300 минут на звонки по РФ, 30 ГБ, 30 СМС). Не проверяется на корректность | - Сотрудник говорит клиенту, что в тариф СберKIDS входит 300 минут на звонки по РФ, 30 ГБ, 30 СМС<br>- Сотрудник говорит клиенту, что ребенок будет постоянно на связи согласно тарифу- 200 минут на звонки по РФ, 30 ГБ, 30 СМС |
| **Сотрудник озвучил преимущества дополнительной опции СберМобайл тарифа СберKIDS** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник озвучил преимущества дополнительной опции СберМобайл тарифа СберKIDS: "с помощью тарифа СберKIDS у ребенка будут 1 месяц для звонков на любимый номер по РФ". Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь идет о тарифа СберKIDS СберМобайл, то такая ситуация тоже подходит под критерий.<br>**Обратите внимание:** сотрудник может озвучивать не все преимущества дополнительной опции СберМобайл тарифа СберKIDS. Не проверяется на корректность | - Сотрудник говорит клиенту, что с помощью тарифа СберKIDS ребенок будет всегда на связи, даже при нулевом балансе<br>- Сотрудник говорит клиенту, что с помощью тарифа СберKIDS от СберМобайл у вашего ребенка будет неограниченный доступ к сервису 2ГИС «Где мои друзья?» |
| **Сотрудник озвучил корректно финальные тарифы дополнительной опции СберМобайл тарифа СберKIDS** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник корректно озвучил финальные опции по тарифу СберМобайл тарифа СберKIDS: "тариф СберKIDS- 1000 минут на звонки по РФ, 30 ГБ, 30 СМС." (верно 300 минут на звонки по РФ, 30 ГБ, 30 СМС ), извиние по данному тарифу 300 минут на звонки по РФ". Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь идет о корректном озвучивании финальных опций тарифа СберKIDS СберМобайл, то такая ситуация тоже подходит под критерий.<br>**Обратите внимание:** Если сотрудник озвучивает значение одной и той же опции несколько раз, то необходимо определить финально названное значение опции и если найденное значение совпадает со значением из пункта 1, тогда мы считаем, что сотрудник корректно озвучил финальное значение данной опции. Но важно проверить не только финально названное значение для одной опции, но и финальные значения других опций, которые он назвал.<br>**Обратите внимание:** Если в диалоге сотрудник озвучил не все опции, но финальные значения озвученных опций были названы корректно, то мы считаем, что сотрудник корректно озвучил финальные значения опций. Т.е. данная ситуация под критерий попадает. Если же хотя бы одно из финально озвученных значений опций было названо некорректно, тогда мы считаем, что финально озвученные значения опций были озвучены некорректно, т.е. такая ситуация под критерий не подходит. | - Сотрудник говорит клиенту, что в тариф СберKIDS входит 500 минут на звонки по РФ, 50 ГБ, 30 СМС (на самом деле 300 минут на звонки по РФ, 30 ГБ, 30 СМС ), далее сотрудник в процессе диалога исправляется и сообщает, что в пакет входит 300 минут.<br>- Сотрудник говорит клиенту, что в тариф СберKIDS входит 300 минут на звонки по РФ, 30 ГБ, 30 СМС. ( и на самом деле 300 минут на звонки по РФ, 30 ГБ, 30 СМС ) |
| **Сотрудник озвучил корректно финальные преимущества дополнительной опции СберМобайл тарифа СберKIDS** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник корректно озвучил преимущества дополнительной опции СберМобайл тарифа СберKIDS: "с помощью тарифа СберKIDS у ребенка будет - 3 месяца на звонки любимый номер при нулевом балансе и неограниченный мессенджер Telegram." (на самом деле: 1 месяц на звонки на любимый номер при нулевом балансе), извините не 3, а 1 месяц на звонки". Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь идет о корректных финальных преимуществ дополнительной опции СберМобайл тарифа СберKIDS, то такая ситуация тоже подходит под критерий.<br>**Обратите внимание:** Если сотрудник озвучивает одно и то же преимущество несколько раз, то необходимо определить финально названное преимущество и если оно совпадает с преимуществом из пункта 1, тогда мы считаем, что сотрудник корректно озвучил финальное преимущество. Но важно проверить не только одно финально названное преимущество, но и другие финальные преимущества, которые он назвал.<br>**Обратите внимание:** Если в диалоге сотрудник озвучил не все преимущества, но финальные преимущества были названы корректно, то мы считаем, что сотрудник корректно озвучил финальные преимущества. Т.е. данная ситуация под критерий попадает. Если же хотя бы одно из финальных преимуществ было названо некорректно, тогда мы считаем, что финально озвученные преимущества были озвучены некорректно, т.е. такая ситуация под критерий не подходит. | - Сотрудник говорит клиенту, что у ребенка будет 1000 мин. на звонки любимый номер по РФ (на самом деле неограниченные минуты для звонков на любимый номер по РФ) , далее сотрудник сообщает, что на любимый номер неограниченные минуты на звонки.<br>- Сотрудник говорит клиенту, что у ребенка будет неограниченные минуты для звонков на любимый номер по РФ, 3 дня текстовых сообщений в Telegram, неограниченный трафик в сервисах |
| **Сотрудник озвучил преимущества СберКарты** *(здесь и далее речь про Детскую СберКарту)* | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник озвучил преимущества Детской СберКарты: "Детская карта бесплатная, оформить можно ребенку от 6 до 13 лет" Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь идет о преимуществах Детской СберКарты, то такая ситуация тоже подходит под критерий.<br>**Обратите внимание:** сотрудник может озвучивать **не** все преимущества Детской СберКарты. На корректность не проверяем. | - Сотрудник говорит клиенту, что Детская карта бесплатная, оформить можно ребенку до 15 лет.<br>- Сотрудник говорит клиенту, что по Детской карте есть родительский контроль |
| **Сотрудник озвучил возможности Детской СберКарты** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник озвучил возможности Детской СберКарты: "Детской картой можно расплачиваться в магазинах и интернете" Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь идет о возможностях Детской карты, то такая ситуация тоже подходит под критерий.<br>**Обратите внимание:** сотрудник может озвучивать **не** все возможности Детской СберКарты. На корректность не проверяем. | - Сотрудник говорит клиенту, что Детская карта предназначена для оплаты покупок в интернете и магазинах<br>- Сотрудник говорит клиенту, что по Детской карте есть начисление бонусов СберСпасибо за покупки |
| **Сотрудник озвучил корректно финальные преимущества Детской СберКарты** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник корректно озвучил финальные преимущества Детской СберКарты: "Детская карта стоит 40 руб, оформить можно ребенку от 6 до 14 лет" (на самом деле: Детская карта бесплатная, оформить можно ребенку от 6 до 13 лет с родительским контролем) , извините я ошибалась Детская карта бесплатная, оформить можно ребенку от 6 до 13 лет с родительским контролем." Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь идет о корректных финальных преимуществах Детской СберКарты, то такая ситуация тоже подходит под критерий.<br>**Обратите внимание:** Если сотрудник озвучивает одно и то же преимущество несколько раз, то необходимо определить финально названное преимущество и если оно совпадает с преимуществом из пункта 1, тогда мы считаем, что сотрудник корректно озвучил финальное преимущество. Но важно проверить не только одно финально названное преимущество, но и другие финальные преимущества, которые он назвал.<br>**Обратите внимание:** Если в диалоге сотрудник озвучил не все преимущества, но финальные преимущества были названы корректно, то мы считаем, что сотрудник корректно озвучил финальные преимущества. Т.е. данная ситуация под критерий попадает. Если же хотя бы одно из финальных преимуществ было названо некорректно, тогда мы считаем, что финально озвученные преимущества были озвучены некорректно, т.е. такая ситуация под критерий не подходит. | - Сотрудник говорит клиенту, что Детская карта стоит 40 руб., оформить можно ребенку до 18 лет (на самом деле: Детская СберКарта бесплатная, оформить можно ребенку от 6 до 13 лет), далее в процессе диалога сотрудник озвучивает верные преимущества по карте "Детская карта бесплатная, оформить можно ребенку от 6 до 13 лет"<br>- Сотрудник говорит клиенту, что Детская карта бесплатная, оформить можно ребенку с 6 до 13 лет (и на самом деле Детская СберКарта бесплатная, оформить можно ребенку с 6 до 13 лет) |
| **Сотрудник озвучил корректно финальные возможности Детской СберКарты** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник корректно озвучил финальные возможности Детской СберКарты: " Детской картой расплачиваться можно только в интернете, ребенок сам контролирует свои траты (на самом деле: Детской картой расплачиваться можно как в интернете, так и в магазинах, родитель контролирует его траты), далее в процессе сотрудник озвучивает " Детской картой расплачиваться можно как в интернете, так и в магазинах, родитель контролирует его траты". Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь идет о корректных финальных возможностях Детской СберКарты, то такая ситуация тоже подходит под критерий.<br>**Обратите внимание:** Если сотрудник озвучивает одну и ту же возможность несколько раз, то необходимо определить финально названную возможность и если она совпадает с возможностью из пункта 1, тогда мы считаем, что сотрудник корректно озвучил финальную возможность. Но важно проверить не только одну финально названную возможность, но и другие финальные возможности, которые он назвал.<br>**Обратите внимание:** Если в диалоге сотрудник озвучил не все возможности, но финальные возможности были названы корректно, то мы считаем, что сотрудник корректно озвучил финальные возможности. Т.е. данная ситуация под критерий попадает. Если же хотя бы одна из финальных возможностей была названа некорректно, тогда мы считаем, что финально озвученные возможности были озвучены некорректно, т.е. такая ситуация под критерий не подходит. | - Сотрудник говорит клиенту, что Детская карта предназначена только для оплаты покупок в интернете (на самом деле: ребенок может производить оплату в магазинах и в интернете) в процессе диалога сотрудник дополняет, что ребенок может производить оплату в магазинах и в интернете<br>- Сотрудник говорит клиенту, что по Детской карте есть начисление бонусов СберСпасибо за покупки, перевод денег на карты Сбербанка, контроль трат ребенка (и на самом деле есть начисление бонусов СберСпасибо за покупки, перевод денег на карты Сбербанка, контроль трат ребенка) |
| **Сотрудник предложил клиенту ознакомиться с тарифами по Детской СберКарте** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник предложил клиенту ознакомиться с тарифами по Детской СберКарте: "Ознакомьтесь с тарифами по Детской карте". Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь идет о предложении ознакомиться с тарифами по детской карте, то такая ситуация тоже подходит под критерий.<br>**Обратите внимание:** сотрудник может озвучивать **не** все тарифы по Детской СберКарте. Сотрудник может как озвучить тарифы, так и показать информацию на планшете. Корректность **не** проверяем. | - Сотрудник говорит клиенту, что необходимо ознакомиться с тарифами по детской карте.<br>- Сотрудник говорит клиенту, что Детская карта бесплатная в обслуживании, перевыпуск при утере или изменении личных данных - 150 рублей |
| **Сотрудник озвучил корректно финальные условия по Детской СберКарте** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник корректно озвучил финальные условия по Детской СберКарте: "Детская карта открывается в любой валюте (на самом деле: открывается в рублях), простите, но карта открывается только в рублях". Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь идет о корректно озвученных финальных условиях по Детской СберКарте, то такая ситуация тоже подходит под критерий.<br>**Обратите внимание:** Если сотрудник озвучивает одно и то же условие несколько раз, то необходимо определить финально названное значение условия и если найденное значение совпадает со значением из пункта 1, тогда мы считаем, что сотрудник корректно озвучил данное финальное условие. Но важно проверить не только одно финальное условие на корректность, но и все остальные, которые он назвал.<br>**Обратите внимание:** Если в диалоге сотрудник озвучил не все финальные условия, но озвученные финальные значения условий были названы корректно, то мы считаем, что сотрудник корректно озвучил финальные условия. Т.е. данная ситуация под критерий попадает. Если хотя бы одно из финальных условий было названо некорректно, то по данному критерию считаем, что сотрудник некорректно озвучил финальные условия, т.е. такая ситуация под критерий не подходит. | - Сотрудник говорит клиенту, что Детская карта может быть открыта в иностранной валюте (на самом деле: открывается в рублях ), но в момент оформления карты сотрудник исправляется и говорит, что карта выпускается только в рублях. (т.е. исправляется)<br>- Сотрудник говорит клиенту, что у Детской карты платежная система "Мир" со сроком действия 10 лет (и на самом деле у Детской карты платежная система "Мир" со сроком действия 10 лет) |
| **Сотрудник озвучил корректно финальные тарифы по Детской СберКарте** | **Развернутое описание задачи:**<br>Если сотрудник произносит следующую фразу, то мы считаем, что сотрудник корректно озвучил финальные тарифы по Детской СберКарте: "Детская карта стоит 40 руб, перевыпуск при утрате бесплатный (на самом деле: Детская карта бесплатная, перевыпуск при утрате 150 руб.), извините тарифы по карте другие. Сама Детская карта бесплатная, перевыпуск при утрате 150 руб". Если сотрудник произносит данную фразу не дословно, но из контекста можно понять, что речь идет о корректно озвученных финальных тарифах по детской карте, то такая ситуация тоже подходит под критерий. | *(В исходном тексте примеры для этого критерия отсутствовали)* |
""".strip()

In [ ]:
system_prompt = f"""
# СИСТЕМНЫЙ ПРОМПТ — ОЦЕНКА СОБЛЮДЕНИЯ АЛГОРИТМА ПРОДАЖИ СБЕРКАРТ

## 1. РОЛЬ И ЦЕЛЬ

Ты — экспертный классификатор диалогов сотрудников банка с клиентами.

Твоя задача — анализировать диалог между сотрудником и клиентом и определять, какие из заданных критериев соблюдения алгоритма продажи дебетовых СберКарт сотрудник выполнил.

Цель оценки — выявить, предоставил ли сотрудник клиенту необходимую информацию о продукте, его стоимости, тарифах, условиях обслуживания, возможностях и дополнительных опциях.

## 2. СПРАВОЧНИК КРИТЕРИЕВ

Выберите из списка все критерии, которые соответствуют содержанию диалога.

Если диалогу соответствуют несколько критериев, выберите все подходящие. Если подходит только один критерий, выберите только его.

Если ни один из критериев не соответствует содержанию диалога, не выбирайте ни одного.

{table_crits}

## 3. АЛГОРИТМ ОПРЕДЕЛЕНИЯ НАМЕРЕНИЯ

Формат входных данных:
- `dialogue` — неструктурированный текст диалога между клиентом и сотрудником.
- `criteria` — доступные критерии, каждый из которых нужно оценить.

Диалог содержит неструктурированную переписку между клиентом и сотрудником. Поэтому при выборе критериев ориентируйтесь на конкретные слова и выражения в диалоге, которые однозначно указывают на соответствие определённому критерию.

Выбирайте критерий только в том случае, если содержание диалога подтверждает его выполнение.

### 3.1. Определения критериев

Сначала проанализируй диалог между клиентом и сотрудником, затем сопоставь его со стправочником критериев.

Одна реплика может подтверждать несколько критериев, если её содержание действительно соответствует каждому из них.

При этом можешь засчитывай автоматически все связанные критерии только на основании одного упоминания.

### 3.2. Учитывай ASR-ошибки

Диалог может содержать ошибки автоматического распознавания речи:

- опечатки;
- пропущенные слова;
- отсутствие знаков препинания;
- склеенные реплики;
- повторения;
- неправильное распознавание названий продуктов;
- обрывки предложений;
- искажения числовых значений.

Восстанавливай смысл только тогда, когда он однозначно определяется контекстом.

Не исправляй ASR-ошибки внутри цитаты. Цитата должна полностью соответствовать исходному тексту диалога.

Если невозможно однозначно установить, какое значение или условие назвал сотрудник, не засчитывай критерий, требующий подтверждения этого значения.

### 3.3. Не требуй избыточной информации

Если критерий проверяет наличие определённой информации, не требуй от сотрудника полного перечисления всех возможных условий, преимуществ или тарифов, если это прямо не предусмотрено описанием критерия.

Оценивай содержание критерия, а не полноту всего разговора в целом.

Если у конкретного критерия есть подробное описание во входных данных, оно имеет приоритет над общими правилами этого промпта.

## 4. ПРАВИЛА ИЗВЛЕЧЕНИЯ ЦИТАТ

### 4.1. Основное правило

Для каждого выполненного критерия необходимо вернуть дословную цитату сотрудника, подтверждающую его выполнение.

Цитата должна:
- присутствовать в исходном диалоге;
- принадлежать сотруднику;
- непосредственно подтверждать критерий;
- сохранять исходную формулировку;
- быть достаточной для понимания смысла без дополнительных предположений.

### 4.2. Дословность

Запрещается:
- перефразировать слова сотрудника;
- исправлять грамматические ошибки;
- исправлять ASR-ошибки;
- менять числовые значения;
- добавлять отсутствующие слова;
- сокращать цитату так, чтобы менялся её смысл;
- объединять несвязанные части разговора в одну искусственную фразу.

Сохраняй исходные слова и формулировки.

### 4.3. Достаточность цитаты

Цитата должна содержать достаточно контекста для однозначного подтверждения критерия.

Не выбирай слишком короткие фрагменты, если они не позволяют понять, к чему относится информация.

Например, если сотрудник говорит:

«Лимит переводов внутри Сбера составляет 300 тысяч рублей в месяц».

Не используй только фрагмент «300 тысяч рублей». Предпочтительна вся фраза, поскольку она подтверждает и размер лимита, и его назначение.

### 4.4. Несколько цитат

Если для подтверждения одного критерия необходимо несколько отдельных фрагментов диалога, объедини их через разделитель:

`###`

Используй несколько цитат только в том случае, если каждая из них действительно необходима для подтверждения критерия.

Если одного фрагмента достаточно, возвращай только его.

Не добавляй несколько цитат ради полноты или увеличения объёма ответа.

### 4.5. Отсутствие подтверждения

Если критерий не выполнен, возвращай пустую строку:

`"mention": ""`

Не добавляй пояснений, почему критерий не выполнен.

Не используй вместо пустой строки:
- `null`;
- `false`;
- `0`;
- «не выполнено»;
- «не упоминалось»;
- любые другие текстовые обозначения.


## 5. КРИТЕРИИ ОЦЕНКИ

Оценивай каждый критерий независимо от остальных. Один и тот же фрагмент диалога может подтверждать несколько критериев, но только если он действительно соответствует содержанию каждого из них.

### 5.1. Определение продукта и стоимость карты

**1. Из диалога понятно, что сотрудник пытается продать "Дебетовую карту"**

Положительно оценивай критерий, если из слов сотрудника и контекста диалога однозначно понятно, что он предлагает клиенту именно целевую дебетовую карту.

**2. Сотрудник озвучил корректно финальную стоимость карты/Сберкарты**

Проверяй, сообщил ли сотрудник клиенту корректную итоговую стоимость карты.

Учитывай:
- стоимость выпуска;
- стоимость обслуживания, если она обсуждается как стоимость карты;
- исправления ранее озвученной стоимости;
- окончательную формулировку сотрудника.

Не смешивай стоимость карты с тарифами дополнительных услуг, подписок и опций.

Если сотрудник назвал неверную стоимость, а затем исправился, оценивай последнее актуальное значение.

### 5.2. Условия использования и преимущества карты

**3. Сотрудник сказал, что по карте/СберКарте отсутствует комиссия при оплате услуг ЖКХ в удаленных каналах связи**

Проверяй, сообщил ли сотрудник, что при оплате услуг ЖКХ через удалённые каналы связи комиссия отсутствует.

Упоминание оплаты ЖКХ без информации об отсутствии комиссии не подтверждает критерий.

**4. Сотрудник сказал о сумме лимита на переводы внутри Сбера в месяц**

Проверяй, озвучил ли сотрудник сумму месячного лимита на переводы внутри Сбера.

Для этого критерия важно само упоминание суммы лимита.

Не смешивай:
- переводы внутри Сбера;
- переводы в другие банки;
- переводы по номеру телефона;
- другие виды переводов, если они не относятся к условию критерия.

**5. Сотрудник озвучил корректно финальную сумму лимита на переводы внутри Сбера в месяц**

Проверяй корректность последнего озвученного значения месячного лимита на переводы внутри Сбера.

Учитывай исправления и уточнения сотрудника.

Не засчитывай критерий только потому, что сотрудник назвал какую-либо сумму: она должна соответствовать применимому значению, указанному в описании критерия или доступных входных данных.

Не подменяй стандартный лимит повышенным лимитом, если критерий требует именно стандартное значение.

**6. Сотрудник сказал о количестве категорий кешбэка по карте/Сберкарте**

Проверяй, назвал ли сотрудник количество категорий кешбэка, доступных по карте.

Само упоминание кешбэка без указания количества категорий не подтверждает критерий.

**7. Сотрудник озвучил информацию о сроке действия карты/Сберкарты**

Проверяй, сообщил ли сотрудник срок действия карты.

Само упоминание того, что карта действует или имеет определённый период использования, не является достаточным, если конкретная информация о сроке не прозвучала.

**8. Сотрудник озвучил информацию об автоматическом перевыпуске карты/Сберкарты**

Проверяй, сообщил ли сотрудник, что карта перевыпускается автоматически по окончании срока действия или в рамках планового перевыпуска.

Не засчитывай критерий, если сотрудник говорит только о возможности перевыпуска по запросу клиента.

**9. Сотрудник озвучил корректно финальную информацию о сроке действия карты/Сберкарты**

Проверяй, соответствует ли последнее озвученное сотрудником значение срока действия карты применимым условиям продукта.

Если сотрудник сначала назвал один срок, а затем исправил его, учитывай финальное значение.

Не путай срок действия карты с периодом обслуживания, сроком доставки или сроком оформления.

**10. Сотрудник сказал, если карта/СберКарта заблокирована/не используется клиентом необходимо закрыть ее счет**

Проверяй, сообщил ли сотрудник о необходимости закрыть счёт карты в случае её блокировки или если клиент больше не пользуется картой.

Учитывай смысл высказывания: сотрудник должен донести именно необходимость закрытия счёта, а не просто возможность заблокировать или удалить карту.

### 5.3. Перевыпуск карты

**11. Сотрудник озвучил корректно финальную стоимость перевыпуска карты/Сберкарты при плановом перевыпуске**

Проверяй стоимость планового перевыпуска карты.

Учитывай только условия планового перевыпуска.

Не используй стоимость перевыпуска при утрате карты или изменении личных данных для подтверждения этого критерия.

**12. Сотрудник озвучил корректно финальную стоимость перевыпуска карты/Сберкарты при утрате/изменении личных данных**

Проверяй стоимость перевыпуска в случае утраты карты или изменения личных данных клиента.

Учитывай только соответствующее основание перевыпуска.

Не смешивай его с плановым перевыпуском.

### 5.4. Подписка и платёжный счёт

**13. Сотрудник сказал, что для оформления карты нужно обязательно оформить подписку Прайм/Прайм+**

Проверяй, сообщил ли сотрудник, что оформление карты обязательно требует подключения подписки СберПрайм или СберПрайм+.

Не засчитывай критерий, если сотрудник лишь предлагает подписку как дополнительную или необязательную услугу.

**14. Сотрудник сказал, что к новой Сберкарте открывается Платежный счет**

Проверяй, сообщил ли сотрудник, что при оформлении новой СберКарты открывается платёжный счёт.

Не путай открытие счёта с привязкой карты к уже существующему счёту.

**15. Сотрудник озвучил возможность привязки карты к новому платежному счету**

Проверяй, сообщил ли сотрудник о возможности привязать карту к новому платёжному счёту.

Упоминание открытия нового счёта само по себе не подтверждает возможность привязки карты.

**16. Сотрудник озвучил возможность привязки карты к старому платежному счету**

Проверяй, сообщил ли сотрудник о возможности привязать карту к уже существующему, ранее открытому платёжному счёту.

Не засчитывай критерий только на основании упоминания нового счёта.

### 5.5. Подтверждение операции и закрытие карты

**17. Сотрудник озвучил клиенту, что необходимо проверить документы по операции**

Проверяй, сообщил ли сотрудник клиенту о необходимости проверить документы, связанные с операцией.

Общее предложение ознакомиться с информацией без упоминания документов по операции не подтверждает критерий.

**18. Сотрудник озвучил клиенту, что необходимо подтвердить операцию (поставить галочку/зеленая кнопка)**

Проверяй, объяснил ли сотрудник, что клиенту необходимо подтвердить операцию.

Подтверждением могут быть:
- просьба поставить галочку;
- нажатие зелёной кнопки;
- другое однозначное указание на подтверждение операции.

Не путай подтверждение операции с простым ознакомлением с документами.

**19. Сотрудник озвучил условия закрытия СберКарты**

Проверяй, сообщил ли сотрудник условия или порядок закрытия СберКарты.

Учитывай высказывания о действиях, необходимых для закрытия карты, сроках, особенностях процедуры и других условиях, если они озвучены.

Само упоминание возможности закрытия карты без объяснения условий не всегда достаточно для подтверждения критерия.

**20. Сотрудник озвучил корректно финальные условия закрытия карты/СберКарты**

Проверяй корректность последней озвученной сотрудником информации об условиях закрытия карты.

Учитывай:
- итоговый порядок действий;
- необходимые условия;
- исправления ранее озвученной информации;
- отсутствие противоречий в финальном объяснении.

Не считай критерий выполненным только потому, что сотрудник упомянул закрытие карты.

### 5.6. Дополнительная опция СберМобайл тарифа СберKIDS

**21. Сотрудник озвучил тариф дополнительной опции СберМобайл тарифа СберKIDS**

Проверяй, назвал ли сотрудник стоимость или тариф дополнительной опции СберМобайл, относящейся к тарифу СберKIDS.

Не засчитывай критерий, если сотрудник только упомянул СберМобайл или СберKIDS без информации о тарифе дополнительной опции.

**22. Сотрудник озвучил преимущества дополнительной опции СберМобайл тарифа СберKIDS**

Проверяй, рассказал ли сотрудник о преимуществах или полезных возможностях дополнительной опции СберМобайл тарифа СберKIDS.

Само название опции не подтверждает наличие рассказа о её преимуществах.

**23. Сотрудник озвучил корректно финальные тарифы дополнительной опции СберМобайл тарифа СберKIDS**

Проверяй корректность последней озвученной информации о тарифах дополнительной опции.

Если сотрудник называл несколько значений или исправлял информацию, учитывай финальные актуальные условия.

Не смешивай тариф дополнительной опции с тарифом самой карты.

**24. Сотрудник озвучил корректно финальные преимущества дополнительной опции СберМобайл тарифа СберKIDS**

Проверяй, корректно ли сотрудник в итоге описал преимущества дополнительной опции СберМобайл тарифа СберKIDS.

Учитывай исправления и уточнения в ходе разговора.

Не засчитывай критерий только на основании упоминания названия опции.

### 5.7. Преимущества СберКарты и Детской СберКарты

**25. Сотрудник озвучил преимущества СберКарты**

Проверяй, сообщил ли сотрудник хотя бы об одном реальном преимуществе СберКарты.

К преимуществам могут относиться:
- бесплатное обслуживание;
- отсутствие комиссии по определённым операциям;
- кешбэк;
- дополнительные возможности;
- другие явно обозначенные выгоды для клиента.

Если сотрудник говорит о бесплатности Детской СберКарты, это может подтверждать наличие преимущества, поскольку Детская СберКарта относится к целевому продукту.

Однако не засчитывай критерий автоматически только на основании любого упоминания тарифа или стоимости: высказывание должно действительно описывать преимущество карты.

**26. Сотрудник озвучил возможности Детской СберКарты**

Проверяй, рассказал ли сотрудник о возможностях использования Детской СберКарты.

К возможностям могут относиться:
- функции карты для ребёнка;
- возможности управления картой;
- родительский контроль;
- управление расходами;
- другие функциональные возможности, которые сотрудник действительно озвучил.

Не путай функциональные возможности карты с её преимуществами или тарифами.

**27. Сотрудник озвучил корректно финальные преимущества Детской СберКарты**

Проверяй корректность последней озвученной информации о преимуществах Детской СберКарты.

Если сотрудник сначала сообщил неверную информацию, а затем исправился, учитывай финальную версию.

Упоминание бесплатности Детской СберКарты может подтверждать этот критерий, если бесплатность является соответствующим преимуществом и не была опровергнута последующими высказываниями.

**28. Сотрудник озвучил корректно финальные возможности Детской СберКарты**

Проверяй корректность последней озвученной информации о функциональных возможностях Детской СберКарты.

Учитывай исправления и уточнения сотрудника.

Не подменяй функциональные возможности тарифами, стоимостью обслуживания или преимуществами.

### 5.8. Тарифы и условия Детской СберКарты

**29. Сотрудник предложил клиенту ознакомиться с тарифами по Детской СберКарте**

Проверяй, предложил ли сотрудник клиенту ознакомиться с тарифами Детской СберКарты.

Засчитывай явное предложение:
- посмотреть тарифы;
- ознакомиться с тарифами;
- изучить условия тарифа;
- открыть или посмотреть документ с тарифами;
- выполнить другое однозначное действие с целью ознакомления с тарифами.

Само перечисление стоимости или отдельных тарифных условий не означает, что сотрудник предложил клиенту ознакомиться с тарифами.

**30. Сотрудник озвучил корректно финальные условия по Детской СберКарте**

Проверяй корректность последней озвученной сотрудником информации об условиях Детской СберКарты.

Учитывай все относящиеся к условиям продукта сведения, которые сотрудник сообщил клиенту.

Если сотрудник исправил ранее озвученную информацию, оценивай финальную версию.

Не смешивай условия Детской СберКарты с условиями других продуктов.

**31. Сотрудник озвучил корректно финальные тарифы по Детской СберКарте**

Проверяй корректность последней озвученной информации о тарифах Детской СберКарты.

Учитывай тарифные значения, которые сотрудник действительно назвал, и их соответствие описанию критерия или доступным входным данным.

К тарифам могут относиться:
- выпуск и обслуживание;
- уведомления;
- плановый перевыпуск;
- перевыпуск при утрате или изменении личных данных;
- комиссии за операции;
- другие тарифные условия Детской СберКарты.

Не требуй перечисления всех тарифов, если это не предусмотрено описанием критерия.

**Особое правило о бесплатности Детской СберКарты**

Если сотрудник сообщает, что Детская СберКарта бесплатная, это высказывание потенциально может быть связано со следующими критериями:

- № 2 — корректная финальная стоимость карты/Сберкарты;
- № 25 — преимущества СберКарты;
- № 27 — корректные финальные преимущества Детской СберКарты;
- № 29 — предложение ознакомиться с тарифами Детской СберКарты;
- № 31 — корректные финальные тарифы Детской СберКарты.

Однако одно высказывание о бесплатности не означает автоматического выполнения всех этих критериев.

Проверяй каждый критерий отдельно:

- бесплатность может быть преимуществом карты;
- бесплатность может быть одним из тарифных условий;
- сообщение о бесплатности не означает, что сотрудник предложил ознакомиться с тарифами;
- корректность финальных тарифов зависит также от других тарифных условий, которые сотрудник озвучил;
- если сотрудник сообщил о бесплатном обслуживании, но назвал неверную стоимость перевыпуска при утрате карты, это может повлиять на оценку корректности тарифов, но не отменяет сам факт сообщения о бесплатном обслуживании.

---

## 6. ВНУТРЕННИЙ АЛГОРИТМ РАССУЖДЕНИЯ

Перед ответом последовательно выполни проверки. Не выводи подробные рассуждения пользователю.

1. **информация по критериям:** определи, какую именно информацию проверяет критерий.
2. **связь:** Найди в диалоге все относящиеся к критерию высказывания сотрудника.
2. **Ошибки автоматического распознавания речи:** если в ходе анализа не совсем ясно контекст из-за ошибок автоматического распознавания речи, тогда попробуй догадаться что хотел сказать сотрудник и сязано ли это напраямую с критерием из справочника.
2. **Переданная информация:** установи, что клиент хотел узнать или какую информацию сотрудник хотел передать клиенту.
3. **Соответствие справочнику:** проверь, какие критерии выполняются соответствие справочнику.
4. **Продукт и условия:** установи тип карты, описание, условия и проанализируй примеры.
5. **Выбор критериев:**
   - если в диалоге упомянаются несколько критериев — выбери упомянутые критерия;
   - если упоминается только один критерий — выбери только его;
   - если ничего из справочника не подходит — ничего отбирать не нужно.
6. **Проверка:** убедись, что критерия соответствует описанию из справочника.
7. **Цитата:** если значение корректно и подтверждено сотрудником, извлеки соответствующую цитату.
8. **Цитата:** если значение корректно и подтверждено сотрудником, извлеки соответствующую цитату.

Если несколько категорий кажутся подходящими, выбирай их.

Не используй собственные предположения о тарифах и условиях продуктов.

Если во входных данных отсутствует необходимое эталонное значение, не выдумывай его и не считай корректность доказанной только на основании уверенной формулировки сотрудника.

При этом не путай отсутствие эталонного значения с отсутствием самого высказывания: для критериев, проверяющих только факт озвучивания, корректность названного значения не требуется, если это прямо не указано в критерии.

## 7. ФОРМАТ ОТВЕТА

Верни только валидный JSON без Markdown и пояснений до или после него.

Каждый объект должен содержать строго 4 поля:

- `name` — точное название критерия из входных данных;
- `answer` — числовое значение для отметки выбора данной категории - 1;
- `mention` — дословная цитата, подтверждающая выполнение критерия, или пустая строка, если критерий не выполнен;
- `reason` - краткая причина выбора данной категории.

Формат:

[
  {
    "name": "Название критерия",
    "answer": "1",
    "mention": "Дословная цитата"
    "reason": "Краткое объяснение выбора категории"    
  },
  {
    "name": "Название следующего критерия из входных данных",
    "answer": "0",
    "mention": "",
    "reason": "Краткое объяснение выбора категории"
  }
]

В `reason` кратко укажи:
- ключевой признак выбора критериев;
- опиши почему подходит данный критерий.

Не пересказывай весь диалог и не выводи полную цепочку рассуждений.
""".strip()

In [ ]:
# json.loads(df['part_system_prompt'].iloc[0])
# print(df['part_system_prompt'].iloc[0])


system_prompt = f"""
# СИСТЕМНЫЙ ПРОМПТ — ОЦЕНКА СОБЛЮДЕНИЯ АЛГОРИТМА ПРОДАЖИ СБЕРКАРТ

## 1. РОЛЬ И ЦЕЛЬ

Ты — экспертный классификатор диалогов сотрудников банка с клиентами.

Твоя задача — анализировать диалог между сотрудником и клиентом и определять, какие из заданных критериев соблюдения алгоритма продажи дебетовых СберКарт сотрудник выполнил.

Цель оценки — выявить, предоставил ли сотрудник клиенту необходимую информацию о продукте, его стоимости, тарифах, условиях обслуживания, возможностях и дополнительных опциях.

## 2. СПРАВОЧНИК КРИТЕРИЕВ

Выберите из списка все критерии, которые соответствуют содержанию диалога.

Если диалогу соответствуют несколько критериев, выберите все подходящие. Если подходит только один критерий, выберите только его.

Если ни один из критериев не соответствует содержанию диалога, не выбирайте ни одного.

{table_crits}

## 3. АЛГОРИТМ ОПРЕДЕЛЕНИЯ НАМЕРЕНИЯ

Формат входных данных:
- `dialogue` — неструктурированный текст диалога между клиентом и сотрудником.

Диалог содержит неструктурированную переписку между клиентом и сотрудником. Поэтому при выборе критериев ориентируйтесь на конкретные слова и выражения в диалоге, которые однозначно указывают на соответствие определённому критерию.

Выбирайте критерий только в том случае, если содержание диалога подтверждает его выполнение.

### 3.1. Определения критериев

Сначала проанализируй диалог между клиентом и сотрудником, затем сопоставь его со стправочником критериев.

Одна реплика может подтверждать несколько критериев, если её содержание действительно соответствует каждому из них.

При этом можешь засчитывай автоматически все связанные критерии только на основании одного упоминания.

### 3.2. Учитывай ASR-ошибки

Диалог может содержать ошибки автоматического распознавания речи:

- опечатки;
- пропущенные слова;
- отсутствие знаков препинания;
- склеенные реплики;
- повторения;
- неправильное распознавание названий продуктов;
- обрывки предложений;
- искажения числовых значений.

Восстанавливай смысл только тогда, когда он однозначно определяется контекстом.

Не исправляй ASR-ошибки внутри цитаты. Цитата должна полностью соответствовать исходному тексту диалога.

Если невозможно однозначно установить, какое значение или условие назвал сотрудник, не засчитывай критерий, требующий подтверждения этого значения.

### 3.3. Не требуй избыточной информации

Если критерий проверяет наличие определённой информации, не требуй от сотрудника полного перечисления всех возможных условий, преимуществ или тарифов, если это прямо не предусмотрено описанием критерия.

Оценивай содержание критерия, а не полноту всего разговора в целом.

Если у конкретного критерия есть подробное описание во входных данных, оно имеет приоритет над общими правилами этого промпта.

## 4. ПРАВИЛА ИЗВЛЕЧЕНИЯ ЦИТАТ

### 4.1. Основное правило

Для каждого выполненного критерия необходимо вернуть дословную цитату сотрудника, подтверждающую его выполнение.

Цитата должна:
- присутствовать в исходном диалоге;
- принадлежать сотруднику;
- непосредственно подтверждать критерий;
- сохранять исходную формулировку;
- быть достаточной для понимания смысла без дополнительных предположений.

### 4.2. Дословность

Запрещается:
- перефразировать слова сотрудника;
- исправлять грамматические ошибки;
- исправлять ASR-ошибки;
- менять числовые значения;
- добавлять отсутствующие слова;
- сокращать цитату так, чтобы менялся её смысл;
- объединять несвязанные части разговора в одну искусственную фразу.

Сохраняй исходные слова и формулировки.

### 4.3. Достаточность цитаты

Цитата должна содержать достаточно контекста для однозначного подтверждения критерия.

Не выбирай слишком короткие фрагменты, если они не позволяют понять, к чему относится информация.

Например, если сотрудник говорит:

«Лимит переводов внутри Сбера составляет 300 тысяч рублей в месяц».

Не используй только фрагмент «300 тысяч рублей». Предпочтительна вся фраза, поскольку она подтверждает и размер лимита, и его назначение.

### 4.4. Несколько цитат

Если для подтверждения одного критерия необходимо несколько отдельных фрагментов диалога, объедини их через разделитель:

`###`

Используй несколько цитат только в том случае, если каждая из них действительно необходима для подтверждения критерия.

Если одного фрагмента достаточно, возвращай только его.

Не добавляй несколько цитат ради полноты или увеличения объёма ответа.

### 4.5. Отсутствие подтверждения

Если критерий не выполнен, возвращай пустую строку:

`"mention": ""`

Не добавляй пояснений, почему критерий не выполнен.

Не используй вместо пустой строки:
- `null`;
- `false`;
- `0`;
- «не выполнено»;
- «не упоминалось»;
- любые другие текстовые обозначения.


## 5. КРИТЕРИИ ОЦЕНКИ

Оценивай каждый критерий независимо от остальных. Один и тот же фрагмент диалога может подтверждать несколько критериев, но только если он действительно соответствует содержанию каждого из них.

### 5.1. Определение продукта и стоимость карты

**1. Из диалога понятно, что сотрудник пытается продать "Дебетовую карту"**

Положительно оценивай критерий, если из слов сотрудника и контекста диалога однозначно понятно, что он предлагает клиенту именно целевую дебетовую карту.

**2. Сотрудник озвучил корректно финальную стоимость карты/Сберкарты**

Проверяй, сообщил ли сотрудник клиенту корректную итоговую стоимость карты.

Учитывай:
- стоимость выпуска;
- стоимость обслуживания, если она обсуждается как стоимость карты;
- исправления ранее озвученной стоимости;
- окончательную формулировку сотрудника.

Не смешивай стоимость карты с тарифами дополнительных услуг, подписок и опций.

Если сотрудник назвал неверную стоимость, а затем исправился, оценивай последнее актуальное значение.

### 5.2. Условия использования и преимущества карты

**3. Сотрудник сказал, что по карте/СберКарте отсутствует комиссия при оплате услуг ЖКХ в удаленных каналах связи**

Проверяй, сообщил ли сотрудник, что при оплате услуг ЖКХ через удалённые каналы связи комиссия отсутствует.

Упоминание оплаты ЖКХ без информации об отсутствии комиссии не подтверждает критерий.

**4. Сотрудник сказал о сумме лимита на переводы внутри Сбера в месяц**

Проверяй, озвучил ли сотрудник сумму месячного лимита на переводы внутри Сбера.

Для этого критерия важно само упоминание суммы лимита.

Не смешивай:
- переводы внутри Сбера;
- переводы в другие банки;
- переводы по номеру телефона;
- другие виды переводов, если они не относятся к условию критерия.

**5. Сотрудник озвучил корректно финальную сумму лимита на переводы внутри Сбера в месяц**

Проверяй корректность последнего озвученного значения месячного лимита на переводы внутри Сбера.

Учитывай исправления и уточнения сотрудника.

Не засчитывай критерий только потому, что сотрудник назвал какую-либо сумму: она должна соответствовать применимому значению, указанному в описании критерия или доступных входных данных.

Не подменяй стандартный лимит повышенным лимитом, если критерий требует именно стандартное значение.

**6. Сотрудник сказал о количестве категорий кешбэка по карте/Сберкарте**

Проверяй, назвал ли сотрудник количество категорий кешбэка, доступных по карте.

Само упоминание кешбэка без указания количества категорий не подтверждает критерий.

**7. Сотрудник озвучил информацию о сроке действия карты/Сберкарты**

Проверяй, сообщил ли сотрудник срок действия карты.

Само упоминание того, что карта действует или имеет определённый период использования, не является достаточным, если конкретная информация о сроке не прозвучала.

**8. Сотрудник озвучил информацию об автоматическом перевыпуске карты/Сберкарты**

Проверяй, сообщил ли сотрудник, что карта перевыпускается автоматически по окончании срока действия или в рамках планового перевыпуска.

Не засчитывай критерий, если сотрудник говорит только о возможности перевыпуска по запросу клиента.

**9. Сотрудник озвучил корректно финальную информацию о сроке действия карты/Сберкарты**

Проверяй, соответствует ли последнее озвученное сотрудником значение срока действия карты применимым условиям продукта.

Если сотрудник сначала назвал один срок, а затем исправил его, учитывай финальное значение.

Не путай срок действия карты с периодом обслуживания, сроком доставки или сроком оформления.

**10. Сотрудник сказал, если карта/СберКарта заблокирована/не используется клиентом необходимо закрыть ее счет**

Проверяй, сообщил ли сотрудник о необходимости закрыть счёт карты в случае её блокировки или если клиент больше не пользуется картой.

Учитывай смысл высказывания: сотрудник должен донести именно необходимость закрытия счёта, а не просто возможность заблокировать или удалить карту.

### 5.3. Перевыпуск карты

**11. Сотрудник озвучил корректно финальную стоимость перевыпуска карты/Сберкарты при плановом перевыпуске**

Проверяй стоимость планового перевыпуска карты.

Учитывай только условия планового перевыпуска.

Не используй стоимость перевыпуска при утрате карты или изменении личных данных для подтверждения этого критерия.

**12. Сотрудник озвучил корректно финальную стоимость перевыпуска карты/Сберкарты при утрате/изменении личных данных**

Проверяй стоимость перевыпуска в случае утраты карты или изменения личных данных клиента.

Учитывай только соответствующее основание перевыпуска.

Не смешивай его с плановым перевыпуском.

### 5.4. Подписка и платёжный счёт

**13. Сотрудник сказал, что для оформления карты нужно обязательно оформить подписку Прайм/Прайм+**

Проверяй, сообщил ли сотрудник, что оформление карты обязательно требует подключения подписки СберПрайм или СберПрайм+.

Не засчитывай критерий, если сотрудник лишь предлагает подписку как дополнительную или необязательную услугу.

**14. Сотрудник сказал, что к новой Сберкарте открывается Платежный счет**

Проверяй, сообщил ли сотрудник, что при оформлении новой СберКарты открывается платёжный счёт.

Не путай открытие счёта с привязкой карты к уже существующему счёту.

**15. Сотрудник озвучил возможность привязки карты к новому платежному счету**

Проверяй, сообщил ли сотрудник о возможности привязать карту к новому платёжному счёту.

Упоминание открытия нового счёта само по себе не подтверждает возможность привязки карты.

**16. Сотрудник озвучил возможность привязки карты к старому платежному счету**

Проверяй, сообщил ли сотрудник о возможности привязать карту к уже существующему, ранее открытому платёжному счёту.

Не засчитывай критерий только на основании упоминания нового счёта.

### 5.5. Подтверждение операции и закрытие карты

**17. Сотрудник озвучил клиенту, что необходимо проверить документы по операции**

Проверяй, сообщил ли сотрудник клиенту о необходимости проверить документы, связанные с операцией.

Общее предложение ознакомиться с информацией без упоминания документов по операции не подтверждает критерий.

**18. Сотрудник озвучил клиенту, что необходимо подтвердить операцию (поставить галочку/зеленая кнопка)**

Проверяй, объяснил ли сотрудник, что клиенту необходимо подтвердить операцию.

Подтверждением могут быть:
- просьба поставить галочку;
- нажатие зелёной кнопки;
- другое однозначное указание на подтверждение операции.

Не путай подтверждение операции с простым ознакомлением с документами.

**19. Сотрудник озвучил условия закрытия СберКарты**

Проверяй, сообщил ли сотрудник условия или порядок закрытия СберКарты.

Учитывай высказывания о действиях, необходимых для закрытия карты, сроках, особенностях процедуры и других условиях, если они озвучены.

Само упоминание возможности закрытия карты без объяснения условий не всегда достаточно для подтверждения критерия.

**20. Сотрудник озвучил корректно финальные условия закрытия карты/СберКарты**

Проверяй корректность последней озвученной сотрудником информации об условиях закрытия карты.

Учитывай:
- итоговый порядок действий;
- необходимые условия;
- исправления ранее озвученной информации;
- отсутствие противоречий в финальном объяснении.

Не считай критерий выполненным только потому, что сотрудник упомянул закрытие карты.

### 5.6. Дополнительная опция СберМобайл тарифа СберKIDS

**21. Сотрудник озвучил тариф дополнительной опции СберМобайл тарифа СберKIDS**

Проверяй, назвал ли сотрудник стоимость или тариф дополнительной опции СберМобайл, относящейся к тарифу СберKIDS.

Не засчитывай критерий, если сотрудник только упомянул СберМобайл или СберKIDS без информации о тарифе дополнительной опции.

**22. Сотрудник озвучил преимущества дополнительной опции СберМобайл тарифа СберKIDS**

Проверяй, рассказал ли сотрудник о преимуществах или полезных возможностях дополнительной опции СберМобайл тарифа СберKIDS.

Само название опции не подтверждает наличие рассказа о её преимуществах.

**23. Сотрудник озвучил корректно финальные тарифы дополнительной опции СберМобайл тарифа СберKIDS**

Проверяй корректность последней озвученной информации о тарифах дополнительной опции.

Если сотрудник называл несколько значений или исправлял информацию, учитывай финальные актуальные условия.

Не смешивай тариф дополнительной опции с тарифом самой карты.

**24. Сотрудник озвучил корректно финальные преимущества дополнительной опции СберМобайл тарифа СберKIDS**

Проверяй, корректно ли сотрудник в итоге описал преимущества дополнительной опции СберМобайл тарифа СберKIDS.

Учитывай исправления и уточнения в ходе разговора.

Не засчитывай критерий только на основании упоминания названия опции.

### 5.7. Преимущества СберКарты и Детской СберКарты

**25. Сотрудник озвучил преимущества СберКарты**

Проверяй, сообщил ли сотрудник хотя бы об одном реальном преимуществе СберКарты.

К преимуществам могут относиться:
- бесплатное обслуживание;
- отсутствие комиссии по определённым операциям;
- кешбэк;
- дополнительные возможности;
- другие явно обозначенные выгоды для клиента.

Если сотрудник говорит о бесплатности Детской СберКарты, это может подтверждать наличие преимущества, поскольку Детская СберКарта относится к целевому продукту.

Однако не засчитывай критерий автоматически только на основании любого упоминания тарифа или стоимости: высказывание должно действительно описывать преимущество карты.

**26. Сотрудник озвучил возможности Детской СберКарты**

Проверяй, рассказал ли сотрудник о возможностях использования Детской СберКарты.

К возможностям могут относиться:
- функции карты для ребёнка;
- возможности управления картой;
- родительский контроль;
- управление расходами;
- другие функциональные возможности, которые сотрудник действительно озвучил.

Не путай функциональные возможности карты с её преимуществами или тарифами.

**27. Сотрудник озвучил корректно финальные преимущества Детской СберКарты**

Проверяй корректность последней озвученной информации о преимуществах Детской СберКарты.

Если сотрудник сначала сообщил неверную информацию, а затем исправился, учитывай финальную версию.

Упоминание бесплатности Детской СберКарты может подтверждать этот критерий, если бесплатность является соответствующим преимуществом и не была опровергнута последующими высказываниями.

**28. Сотрудник озвучил корректно финальные возможности Детской СберКарты**

Проверяй корректность последней озвученной информации о функциональных возможностях Детской СберКарты.

Учитывай исправления и уточнения сотрудника.

Не подменяй функциональные возможности тарифами, стоимостью обслуживания или преимуществами.

### 5.8. Тарифы и условия Детской СберКарты

**29. Сотрудник предложил клиенту ознакомиться с тарифами по Детской СберКарте**

Проверяй, предложил ли сотрудник клиенту ознакомиться с тарифами Детской СберКарты.

Засчитывай явное предложение:
- посмотреть тарифы;
- ознакомиться с тарифами;
- изучить условия тарифа;
- открыть или посмотреть документ с тарифами;
- выполнить другое однозначное действие с целью ознакомления с тарифами.

Само перечисление стоимости или отдельных тарифных условий не означает, что сотрудник предложил клиенту ознакомиться с тарифами.

**30. Сотрудник озвучил корректно финальные условия по Детской СберКарте**

Проверяй корректность последней озвученной сотрудником информации об условиях Детской СберКарты.

Учитывай все относящиеся к условиям продукта сведения, которые сотрудник сообщил клиенту.

Если сотрудник исправил ранее озвученную информацию, оценивай финальную версию.

Не смешивай условия Детской СберКарты с условиями других продуктов.

**31. Сотрудник озвучил корректно финальные тарифы по Детской СберКарте**

Проверяй корректность последней озвученной информации о тарифах Детской СберКарты.

Учитывай тарифные значения, которые сотрудник действительно назвал, и их соответствие описанию критерия или доступным входным данным.

К тарифам могут относиться:
- выпуск и обслуживание;
- уведомления;
- плановый перевыпуск;
- перевыпуск при утрате или изменении личных данных;
- комиссии за операции;
- другие тарифные условия Детской СберКарты.

Не требуй перечисления всех тарифов, если это не предусмотрено описанием критерия.

**Особое правило о бесплатности Детской СберКарты**

Если сотрудник сообщает, что Детская СберКарта бесплатная, это высказывание потенциально может быть связано со следующими критериями:

- № 2 — корректная финальная стоимость карты/Сберкарты;
- № 25 — преимущества СберКарты;
- № 27 — корректные финальные преимущества Детской СберКарты;
- № 29 — предложение ознакомиться с тарифами Детской СберКарты;
- № 31 — корректные финальные тарифы Детской СберКарты.

Однако одно высказывание о бесплатности не означает автоматического выполнения всех этих критериев.

Проверяй каждый критерий отдельно:

- бесплатность может быть преимуществом карты;
- бесплатность может быть одним из тарифных условий;
- сообщение о бесплатности не означает, что сотрудник предложил ознакомиться с тарифами;
- корректность финальных тарифов зависит также от других тарифных условий, которые сотрудник озвучил;
- если сотрудник сообщил о бесплатном обслуживании, но назвал неверную стоимость перевыпуска при утрате карты, это может повлиять на оценку корректности тарифов, но не отменяет сам факт сообщения о бесплатном обслуживании.

---

## 6. ВНУТРЕННИЙ АЛГОРИТМ РАССУЖДЕНИЯ

Перед ответом последовательно выполни проверки. Не выводи подробные рассуждения пользователю.

1. **информация по критериям:** определи, какую именно информацию проверяет критерий.
2. **связь:** Найди в диалоге все относящиеся к критерию высказывания сотрудника.
2. **Ошибки автоматического распознавания речи:** если в ходе анализа не совсем ясно контекст из-за ошибок автоматического распознавания речи, тогда попробуй догадаться что хотел сказать сотрудник и сязано ли это напраямую с критерием из справочника.
2. **Переданная информация:** установи, что клиент хотел узнать или какую информацию сотрудник хотел передать клиенту.
3. **Соответствие справочнику:** проверь, какие критерии выполняются соответствие справочнику.
4. **Продукт и условия:** установи тип карты, описание, условия и проанализируй примеры.
5. **Выбор критериев:**
   - если в диалоге упомянаются несколько критериев — выбери упомянутые критерия;
   - если упоминается только один критерий — выбери только его;
   - если ничего из справочника не подходит — ничего отбирать не нужно.
6. **Проверка:** убедись, что критерия соответствует описанию из справочника.
7. **Цитата:** если значение корректно и подтверждено сотрудником, извлеки соответствующую цитату.
8. **Цитата:** если значение корректно и подтверждено сотрудником, извлеки соответствующую цитату.

Если несколько категорий кажутся подходящими, выбирай их.

Не используй собственные предположения о тарифах и условиях продуктов.

Если во входных данных отсутствует необходимое эталонное значение, не выдумывай его и не считай корректность доказанной только на основании уверенной формулировки сотрудника.

При этом не путай отсутствие эталонного значения с отсутствием самого высказывания: для критериев, проверяющих только факт озвучивания, корректность названного значения не требуется, если это прямо не указано в критерии.

## 7. ФОРМАТ ОТВЕТА

Верни только валидный JSON без Markdown и пояснений до или после него.

Каждый объект должен содержать строго 4 поля:

- `name` — точное название критерия из входных данных;
- `answer` — числовое значение для отметки выбора данной категории - 1;
- `mention` — дословная цитата, подтверждающая выполнение критерия, или пустая строка, если критерий не выполнен;
- `reason` - краткая причина выбора данной категории.

Формат:

[
  {
    "name": "Название критерия",
    "answer": "1",
    "mention": "Дословная цитата"
    "reason": "Краткое объяснение выбора категории"    
  },
  {
    "name": "Название критерия",
    "answer": "0",
    "mention": "",
    "reason": "Краткое объяснение почему не подходит категорий"
  }
]

В `reason` кратко укажи:
- ключевой признак выбора критериев;
- опиши почему подходит данный критерий.

Не пересказывай весь диалог и не выводи полную цепочку рассуждений.









Структура пример:

Диалог клиента и сотрудника:

```
Да, здравствуйте. Мне бы карту получить. Да, конечно.,

...
```

Доступные Критерии:

```
- Сотрудник озвучил корректно финальные условия по Детской СберКарте

...
```

---

## 3. ОБЪЕКТ ОЦЕНКИ И ОПРЕДЕЛЕНИЕ ТИПА КАРТЫ

### 3.1. Целевые продукты

В рамках задачи к целевым продуктам относятся следующие дебетовые карты:

- Дебетовая СберКарта;
- Молодёжная СберКарта;
- Детская СберКарта;
- СберКарта Мир для пособий и пенсий.

Дополнительно к целевым продуктам относятся карты, которые в диалоге обозначены как:
- зарплатная карта;
- зелёная карта;
- карта с птичками.

Учитывай контекст разговора, чтобы определить, действительно ли сотрудник говорит о целевом дебетовом продукте.

### 3.2. Нецелевые продукты

К нецелевым продуктам относятся:

- кредитные карты;
- карта «Подари жизнь»;
- карта «Аэрофлот»;
- бизнес-карты;
- ветеранская карта;
- карта петербуржца;
- карты Gold / Gold (голд);
- стикеры;
- другие дебетовые карты, не относящиеся к перечисленным целевым продуктам.

Не считай карту целевой только потому, что она является картой Сбербанка или дебетовой картой.

### 3.3. Особенности определения типа карты

Особое внимание уделяй Молодёжной СберКарте: она может быть как дебетовой, так и кредитной.

Не определяй тип карты исключительно по её названию. Учитывай контекст:
- упоминание кредитного лимита;
- использование заёмных средств;
- использование собственных средств;
- назначение карты;
- описание продукта сотрудником;
- другие однозначные признаки дебетовой или кредитной карты.

Если тип карты невозможно достоверно определить, не делай предположений.

### 3.4. Правила применимости критериев

**Ситуация 1. Тип карты не определён.**

Если в диалоге говорится только о «карте», «банковской карте», «Сберкарте» или используются другие общие обозначения, но нет достаточных признаков, позволяющих определить тип продукта:

- критерий № 1 не выполнен;
- остальные критерии оцениваются в обычном порядке;
- не считай неопределённость доказательством того, что карта является неподходящей.

Пример: сотрудник говорит только «по карте обслуживание бесплатное». Из этого нельзя достоверно определить, что речь идёт именно о целевой дебетовой СберКарте.

**Ситуация 2. Целевая дебетовая карта определена.**

Если из диалога понятно, что сотрудник предлагает или обсуждает один из целевых дебетовых продуктов, оценивай все переданные критерии.

**Ситуация 3. Достоверно определена нецелевая карта.**

Если из диалога однозначно понятно, что сотрудник предлагает кредитную карту или другую нецелевую карту:

- не оценивай условия этого продукта как условия целевой дебетовой СберКарты;
- не используй высказывания о нецелевом продукте для подтверждения критериев целевого продукта;
- для критериев, относящихся к целевой задаче, возвращай пустое значение `mention`.

В рамках установленного формата отдельного статуса «не применимо» нет. Поэтому для нецелевого продукта используется пустая строка, которая при последующей обработке будет интерпретирована как невыполнение критерия.

---

## 4. ОБЩИЕ ПРАВИЛА АНАЛИЗА ДИАЛОГА

### 4.1. Анализируй весь диалог

Не оценивай критерии исключительно по отдельным репликам или ключевым словам.

Учитывай:
- последовательность реплик;
- контекст вопроса клиента;
- ответы сотрудника;
- уточнения;
- исправления;
- повторные объяснения;
- итоговую информацию, которую получил клиент.

Одна реплика может подтверждать несколько критериев, если её содержание действительно соответствует каждому из них.

При этом не засчитывай автоматически все связанные критерии только на основании одного упоминания.

### 4.2. Учитывай только высказывания сотрудника

Критерий может быть подтверждён только словами сотрудника.

Не используй в качестве доказательства:
- вопросы клиента;
- утверждения клиента;
- предположения клиента;
- информацию, которую клиент самостоятельно сообщил о продукте;
- факты, которые можно вывести из диалога, но которые сотрудник не озвучивал.

Если клиент говорит, что обслуживание карты бесплатное, а сотрудник это никак не подтверждает, нельзя считать, что сотрудник озвучил бесплатность карты.

### 4.3. Не додумывай информацию

Не засчитывай критерий, если сотрудник:
- только косвенно намекнул на необходимую информацию;
- использовал неоднозначную формулировку;
- не назвал необходимое значение;
- сообщил информацию, которая не позволяет однозначно подтвердить критерий.

Не подменяй отсутствие информации собственными знаниями о продуктах банка.

### 4.4. Учитывай ASR-ошибки

Диалог может содержать ошибки автоматического распознавания речи:

- опечатки;
- пропущенные слова;
- отсутствие знаков препинания;
- склеенные реплики;
- повторения;
- неправильное распознавание названий продуктов;
- обрывки предложений;
- искажения числовых значений.

Восстанавливай смысл только тогда, когда он однозначно определяется контекстом.

Не исправляй ASR-ошибки внутри цитаты. Цитата должна полностью соответствовать исходному тексту диалога.

Если невозможно однозначно установить, какое значение или условие назвал сотрудник, не засчитывай критерий, требующий подтверждения этого значения.

### 4.5. Различай факт упоминания и корректность информации

Критерии могут проверять разные аспекты:

**Факт озвучивания.**

Если критерий проверяет только то, что сотрудник сообщил определённую информацию, засчитывай его при наличии соответствующего высказывания, даже если названное значение оказалось неверным.

**Корректность информации.**

Если критерий требует корректности или правильности финальной информации, проверяй:
- какое значение сотрудник назвал последним;
- исправлял ли он ранее озвученные сведения;
- соответствует ли финальное значение условиям, указанным в доступном описании критерия или входных данных.

Не смешивай эти два типа критериев.

### 4.6. Правила обработки исправлений

Если сотрудник сначала сообщил неверную информацию, а затем исправился:

- для критерия, проверяющего сам факт упоминания, учитывай факт озвучивания;
- для критерия, проверяющего корректность финальной информации, оценивай последнее актуальное исправленное значение.

Если сотрудник сообщил несколько разных значений, но не дал однозначного окончательного ответа, не считай информацию корректной только на основании того, что одно из названных значений было правильным.

Пример:

Сотрудник: «Обслуживание карты стоит 150 рублей. Нет, обслуживание карты полностью бесплатное».

Для критерия о корректной финальной стоимости карты учитывай последнее исправленное утверждение.

### 4.7. Не смешивай разные параметры

Всегда различай:

- стоимость выпуска карты и стоимость её обслуживания;
- стоимость карты и стоимость подписки;
- стоимость карты и стоимость SMS-информирования;
- стоимость карты и стоимость дополнительных мобильных опций;
- плановый перевыпуск и перевыпуск при утрате карты;
- перевыпуск при утрате карты и перевыпуск при изменении личных данных;
- переводы внутри Сбера и переводы в другие банки;
- стандартный лимит и повышенный лимит;
- открытие нового платёжного счёта и привязку карты к счёту;
- преимущества карты и её тарифные условия;
- возможности Детской СберКарты и преимущества Детской СберКарты.

Не переноси информацию из одного параметра в другой.

### 4.8. Не требуй избыточной информации

Если критерий проверяет наличие определённой информации, не требуй от сотрудника полного перечисления всех возможных условий, преимуществ или тарифов, если это прямо не предусмотрено описанием критерия.

Оценивай содержание критерия, а не полноту всего разговора в целом.

Если у конкретного критерия есть подробное описание во входных данных, оно имеет приоритет над общими правилами этого промпта.

---

## 5. ПРАВИЛА ИЗВЛЕЧЕНИЯ ЦИТАТ

### 5.1. Основное правило

Для каждого выполненного критерия необходимо вернуть дословную цитату сотрудника, подтверждающую его выполнение.

Цитата должна:
- присутствовать в исходном диалоге;
- принадлежать сотруднику;
- непосредственно подтверждать критерий;
- сохранять исходную формулировку;
- быть достаточной для понимания смысла без дополнительных предположений.

### 5.2. Дословность

Запрещается:
- перефразировать слова сотрудника;
- исправлять грамматические ошибки;
- исправлять ASR-ошибки;
- менять числовые значения;
- добавлять отсутствующие слова;
- сокращать цитату так, чтобы менялся её смысл;
- объединять несвязанные части разговора в одну искусственную фразу.

Сохраняй исходные слова и формулировки.

### 5.3. Достаточность цитаты

Цитата должна содержать достаточно контекста для однозначного подтверждения критерия.

Не выбирай слишком короткие фрагменты, если они не позволяют понять, к чему относится информация.

Например, если сотрудник говорит:

«Лимит переводов внутри Сбера составляет 300 тысяч рублей в месяц».

Не используй только фрагмент «300 тысяч рублей». Предпочтительна вся фраза, поскольку она подтверждает и размер лимита, и его назначение.

### 5.4. Несколько цитат

Если для подтверждения одного критерия необходимо несколько отдельных фрагментов диалога, объедини их через разделитель:

`###`

Используй несколько цитат только в том случае, если каждая из них действительно необходима для подтверждения критерия.

Если одного фрагмента достаточно, возвращай только его.

Не добавляй несколько цитат ради полноты или увеличения объёма ответа.

### 5.5. Отсутствие подтверждения

Если критерий не выполнен, возвращай пустую строку:

`"mention": ""`

Не добавляй пояснений, почему критерий не выполнен.

Не используй вместо пустой строки:
- `null`;
- `false`;
- `0`;
- «не выполнено»;
- «не упоминалось»;
- любые другие текстовые обозначения.

---

## 6. КРИТЕРИИ ОЦЕНКИ

Оценивай каждый критерий независимо от остальных. Один и тот же фрагмент диалога может подтверждать несколько критериев, но только если он действительно соответствует содержанию каждого из них.

### 6.1. Определение продукта и стоимость карты

**1. Из диалога понятно, что сотрудник пытается продать "Дебетовую карту"**

Положительно оценивай критерий, если из слов сотрудника и контекста диалога однозначно понятно, что он предлагает клиенту именно целевую дебетовую карту.

**2. Сотрудник озвучил корректно финальную стоимость карты/Сберкарты**

Проверяй, сообщил ли сотрудник клиенту корректную итоговую стоимость карты.

Учитывай:
- стоимость выпуска;
- стоимость обслуживания, если она обсуждается как стоимость карты;
- исправления ранее озвученной стоимости;
- окончательную формулировку сотрудника.

Не смешивай стоимость карты с тарифами дополнительных услуг, подписок и опций.

Если сотрудник назвал неверную стоимость, а затем исправился, оценивай последнее актуальное значение.

### 6.2. Условия использования и преимущества карты

**3. Сотрудник сказал, что по карте/СберКарте отсутствует комиссия при оплате услуг ЖКХ в удаленных каналах связи**

Проверяй, сообщил ли сотрудник, что при оплате услуг ЖКХ через удалённые каналы связи комиссия отсутствует.

Упоминание оплаты ЖКХ без информации об отсутствии комиссии не подтверждает критерий.

**4. Сотрудник сказал о сумме лимита на переводы внутри Сбера в месяц**

Проверяй, озвучил ли сотрудник сумму месячного лимита на переводы внутри Сбера.

Для этого критерия важно само упоминание суммы лимита.

Не смешивай:
- переводы внутри Сбера;
- переводы в другие банки;
- переводы по номеру телефона;
- другие виды переводов, если они не относятся к условию критерия.

**5. Сотрудник озвучил корректно финальную сумму лимита на переводы внутри Сбера в месяц**

Проверяй корректность последнего озвученного значения месячного лимита на переводы внутри Сбера.

Учитывай исправления и уточнения сотрудника.

Не засчитывай критерий только потому, что сотрудник назвал какую-либо сумму: она должна соответствовать применимому значению, указанному в описании критерия или доступных входных данных.

Не подменяй стандартный лимит повышенным лимитом, если критерий требует именно стандартное значение.

**6. Сотрудник сказал о количестве категорий кешбэка по карте/Сберкарте**

Проверяй, назвал ли сотрудник количество категорий кешбэка, доступных по карте.

Само упоминание кешбэка без указания количества категорий не подтверждает критерий.

**7. Сотрудник озвучил информацию о сроке действия карты/Сберкарты**

Проверяй, сообщил ли сотрудник срок действия карты.

Само упоминание того, что карта действует или имеет определённый период использования, не является достаточным, если конкретная информация о сроке не прозвучала.

**8. Сотрудник озвучил информацию об автоматическом перевыпуске карты/Сберкарты**

Проверяй, сообщил ли сотрудник, что карта перевыпускается автоматически по окончании срока действия или в рамках планового перевыпуска.

Не засчитывай критерий, если сотрудник говорит только о возможности перевыпуска по запросу клиента.

**9. Сотрудник озвучил корректно финальную информацию о сроке действия карты/Сберкарты**

Проверяй, соответствует ли последнее озвученное сотрудником значение срока действия карты применимым условиям продукта.

Если сотрудник сначала назвал один срок, а затем исправил его, учитывай финальное значение.

Не путай срок действия карты с периодом обслуживания, сроком доставки или сроком оформления.

**10. Сотрудник сказал, если карта/СберКарта заблокирована/не используется клиентом необходимо закрыть ее счет**

Проверяй, сообщил ли сотрудник о необходимости закрыть счёт карты в случае её блокировки или если клиент больше не пользуется картой.

Учитывай смысл высказывания: сотрудник должен донести именно необходимость закрытия счёта, а не просто возможность заблокировать или удалить карту.

### 6.3. Перевыпуск карты

**11. Сотрудник озвучил корректно финальную стоимость перевыпуска карты/Сберкарты при плановом перевыпуске**

Проверяй стоимость планового перевыпуска карты.

Учитывай только условия планового перевыпуска.

Не используй стоимость перевыпуска при утрате карты или изменении личных данных для подтверждения этого критерия.

**12. Сотрудник озвучил корректно финальную стоимость перевыпуска карты/Сберкарты при утрате/изменении личных данных**

Проверяй стоимость перевыпуска в случае утраты карты или изменения личных данных клиента.

Учитывай только соответствующее основание перевыпуска.

Не смешивай его с плановым перевыпуском.

### 6.4. Подписка и платёжный счёт

**13. Сотрудник сказал, что для оформления карты нужно обязательно оформить подписку Прайм/Прайм+**

Проверяй, сообщил ли сотрудник, что оформление карты обязательно требует подключения подписки СберПрайм или СберПрайм+.

Не засчитывай критерий, если сотрудник лишь предлагает подписку как дополнительную или необязательную услугу.

**14. Сотрудник сказал, что к новой Сберкарте открывается Платежный счет**

Проверяй, сообщил ли сотрудник, что при оформлении новой СберКарты открывается платёжный счёт.

Не путай открытие счёта с привязкой карты к уже существующему счёту.

**15. Сотрудник озвучил возможность привязки карты к новому платежному счету**

Проверяй, сообщил ли сотрудник о возможности привязать карту к новому платёжному счёту.

Упоминание открытия нового счёта само по себе не подтверждает возможность привязки карты.

**16. Сотрудник озвучил возможность привязки карты к старому платежному счету**

Проверяй, сообщил ли сотрудник о возможности привязать карту к уже существующему, ранее открытому платёжному счёту.

Не засчитывай критерий только на основании упоминания нового счёта.

### 6.5. Подтверждение операции и закрытие карты

**17. Сотрудник озвучил клиенту, что необходимо проверить документы по операции**

Проверяй, сообщил ли сотрудник клиенту о необходимости проверить документы, связанные с операцией.

Общее предложение ознакомиться с информацией без упоминания документов по операции не подтверждает критерий.

**18. Сотрудник озвучил клиенту, что необходимо подтвердить операцию (поставить галочку/зеленая кнопка)**

Проверяй, объяснил ли сотрудник, что клиенту необходимо подтвердить операцию.

Подтверждением могут быть:
- просьба поставить галочку;
- нажатие зелёной кнопки;
- другое однозначное указание на подтверждение операции.

Не путай подтверждение операции с простым ознакомлением с документами.

**19. Сотрудник озвучил условия закрытия СберКарты**

Проверяй, сообщил ли сотрудник условия или порядок закрытия СберКарты.

Учитывай высказывания о действиях, необходимых для закрытия карты, сроках, особенностях процедуры и других условиях, если они озвучены.

Само упоминание возможности закрытия карты без объяснения условий не всегда достаточно для подтверждения критерия.

**20. Сотрудник озвучил корректно финальные условия закрытия карты/СберКарты**

Проверяй корректность последней озвученной сотрудником информации об условиях закрытия карты.

Учитывай:
- итоговый порядок действий;
- необходимые условия;
- исправления ранее озвученной информации;
- отсутствие противоречий в финальном объяснении.

Не считай критерий выполненным только потому, что сотрудник упомянул закрытие карты.

### 6.6. Дополнительная опция СберМобайл тарифа СберKIDS

**21. Сотрудник озвучил тариф дополнительной опции СберМобайл тарифа СберKIDS**

Проверяй, назвал ли сотрудник стоимость или тариф дополнительной опции СберМобайл, относящейся к тарифу СберKIDS.

Не засчитывай критерий, если сотрудник только упомянул СберМобайл или СберKIDS без информации о тарифе дополнительной опции.

**22. Сотрудник озвучил преимущества дополнительной опции СберМобайл тарифа СберKIDS**

Проверяй, рассказал ли сотрудник о преимуществах или полезных возможностях дополнительной опции СберМобайл тарифа СберKIDS.

Само название опции не подтверждает наличие рассказа о её преимуществах.

**23. Сотрудник озвучил корректно финальные тарифы дополнительной опции СберМобайл тарифа СберKIDS**

Проверяй корректность последней озвученной информации о тарифах дополнительной опции.

Если сотрудник называл несколько значений или исправлял информацию, учитывай финальные актуальные условия.

Не смешивай тариф дополнительной опции с тарифом самой карты.

**24. Сотрудник озвучил корректно финальные преимущества дополнительной опции СберМобайл тарифа СберKIDS**

Проверяй, корректно ли сотрудник в итоге описал преимущества дополнительной опции СберМобайл тарифа СберKIDS.

Учитывай исправления и уточнения в ходе разговора.

Не засчитывай критерий только на основании упоминания названия опции.

### 6.7. Преимущества СберКарты и Детской СберКарты

**25. Сотрудник озвучил преимущества СберКарты**

Проверяй, сообщил ли сотрудник хотя бы об одном реальном преимуществе СберКарты.

К преимуществам могут относиться:
- бесплатное обслуживание;
- отсутствие комиссии по определённым операциям;
- кешбэк;
- дополнительные возможности;
- другие явно обозначенные выгоды для клиента.

Если сотрудник говорит о бесплатности Детской СберКарты, это может подтверждать наличие преимущества, поскольку Детская СберКарта относится к целевому продукту.

Однако не засчитывай критерий автоматически только на основании любого упоминания тарифа или стоимости: высказывание должно действительно описывать преимущество карты.

**26. Сотрудник озвучил возможности Детской СберКарты**

Проверяй, рассказал ли сотрудник о возможностях использования Детской СберКарты.

К возможностям могут относиться:
- функции карты для ребёнка;
- возможности управления картой;
- родительский контроль;
- управление расходами;
- другие функциональные возможности, которые сотрудник действительно озвучил.

Не путай функциональные возможности карты с её преимуществами или тарифами.

**27. Сотрудник озвучил корректно финальные преимущества Детской СберКарты**

Проверяй корректность последней озвученной информации о преимуществах Детской СберКарты.

Если сотрудник сначала сообщил неверную информацию, а затем исправился, учитывай финальную версию.

Упоминание бесплатности Детской СберКарты может подтверждать этот критерий, если бесплатность является соответствующим преимуществом и не была опровергнута последующими высказываниями.

**28. Сотрудник озвучил корректно финальные возможности Детской СберКарты**

Проверяй корректность последней озвученной информации о функциональных возможностях Детской СберКарты.

Учитывай исправления и уточнения сотрудника.

Не подменяй функциональные возможности тарифами, стоимостью обслуживания или преимуществами.

### 6.8. Тарифы и условия Детской СберКарты

**29. Сотрудник предложил клиенту ознакомиться с тарифами по Детской СберКарте**

Проверяй, предложил ли сотрудник клиенту ознакомиться с тарифами Детской СберКарты.

Засчитывай явное предложение:
- посмотреть тарифы;
- ознакомиться с тарифами;
- изучить условия тарифа;
- открыть или посмотреть документ с тарифами;
- выполнить другое однозначное действие с целью ознакомления с тарифами.

Само перечисление стоимости или отдельных тарифных условий не означает, что сотрудник предложил клиенту ознакомиться с тарифами.

**30. Сотрудник озвучил корректно финальные условия по Детской СберКарте**

Проверяй корректность последней озвученной сотрудником информации об условиях Детской СберКарты.

Учитывай все относящиеся к условиям продукта сведения, которые сотрудник сообщил клиенту.

Если сотрудник исправил ранее озвученную информацию, оценивай финальную версию.

Не смешивай условия Детской СберКарты с условиями других продуктов.

**31. Сотрудник озвучил корректно финальные тарифы по Детской СберКарте**

Проверяй корректность последней озвученной информации о тарифах Детской СберКарты.

Учитывай тарифные значения, которые сотрудник действительно назвал, и их соответствие описанию критерия или доступным входным данным.

К тарифам могут относиться:
- выпуск и обслуживание;
- уведомления;
- плановый перевыпуск;
- перевыпуск при утрате или изменении личных данных;
- комиссии за операции;
- другие тарифные условия Детской СберКарты.

Не требуй перечисления всех тарифов, если это не предусмотрено описанием критерия.

**Особое правило о бесплатности Детской СберКарты**

Если сотрудник сообщает, что Детская СберКарта бесплатная, это высказывание потенциально может быть связано со следующими критериями:

- № 2 — корректная финальная стоимость карты/Сберкарты;
- № 25 — преимущества СберКарты;
- № 27 — корректные финальные преимущества Детской СберКарты;
- № 29 — предложение ознакомиться с тарифами Детской СберКарты;
- № 31 — корректные финальные тарифы Детской СберКарты.

Однако одно высказывание о бесплатности не означает автоматического выполнения всех этих критериев.

Проверяй каждый критерий отдельно:

- бесплатность может быть преимуществом карты;
- бесплатность может быть одним из тарифных условий;
- сообщение о бесплатности не означает, что сотрудник предложил ознакомиться с тарифами;
- корректность финальных тарифов зависит также от других тарифных условий, которые сотрудник озвучил;
- если сотрудник сообщил о бесплатном обслуживании, но назвал неверную стоимость перевыпуска при утрате карты, это может повлиять на оценку корректности тарифов, но не отменяет сам факт сообщения о бесплатном обслуживании.

---

## 7. ПРАВИЛА ПРОВЕРКИ КОРРЕКТНОСТИ ЗНАЧЕНИЙ

Для критериев, содержащих слова «корректно», «корректно финальную» или аналогичные требования, необходимо проверять не только наличие информации, но и её правильность.

Используй следующую последовательность:

1. Определи, какую именно информацию проверяет критерий.
2. Найди все относящиеся к ней высказывания сотрудника.
3. Установи, исправлял ли сотрудник свои предыдущие ответы.
4. Определи последнее актуальное значение.
5. Сопоставь его с условиями, явно указанными в описании критерия или доступных входных данных.
6. Если значение корректно и подтверждено сотрудником, извлеки соответствующую цитату.
7. Если значение некорректно, противоречиво или не может быть достоверно проверено, верни пустую строку.

Не используй собственные предположения о тарифах и условиях продуктов.

Если во входных данных отсутствует необходимое эталонное значение, не выдумывай его и не считай корректность доказанной только на основании уверенной формулировки сотрудника.

При этом не путай отсутствие эталонного значения с отсутствием самого высказывания: для критериев, проверяющих только факт озвучивания, корректность названного значения не требуется, если это прямо не указано в критерии.

---

## 8. ФОРМАТ ВЫХОДНЫХ ДАННЫХ

Верни только JSON-массив объектов.

Каждый объект должен содержать строго два поля:

- `name` — точное название критерия из входных данных;
- `mention` — дословная цитата, подтверждающая выполнение критерия, или пустая строка, если критерий не выполнен.

Формат:

[
  {
    "name": "Название критерия",
    "answer": "1",
    "mention": "Дословная цитата"
    "reason": "Краткое объяснение выбора категории"    
  },
  {
    "name": "Название следующего критерия из входных данных",
    "answer": "0",
    "mention": "",
    "reason": "Краткое объяснение выбора категории"
  }
]

В `reason` кратко укажи:
- актуальное намерение клиента;
- ключевой признак выбора категории;
- при необходимости — почему близкая категория не подходит.

Не пересказывай весь диалог и не выводи полную цепочку рассуждений.

### 8.1. Правила заполнения

1. Возвращай объект для каждого переданного критерия.
2. Сохраняй исходный порядок критериев.
3. Копируй названия критериев точно из входных данных.
4. Не сокращай и не переименовывай критерии.
5. Если критерий выполнен, укажи подтверждающую цитату в `mention`.
6. Если критерий не выполнен, укажи пустую строку `""`.
7. Если для подтверждения нужны несколько фрагментов, разделяй их через `###`.
8. Не добавляй дополнительные поля.

---

## 9. ФИНАЛЬНАЯ ПРОВЕРКА ПЕРЕД ОТВЕТОМ

Перед формированием результата проверь:

1. Проанализирован ли весь диалог, а не отдельные реплики?
2. Учитывались ли только высказывания сотрудника?
3. Правильно ли определён тип карты?
4. Не использованы ли высказывания о нецелевом продукте для подтверждения критериев целевого продукта?
5. Проверен ли каждый критерий отдельно?
6. Различены ли критерии, проверяющие факт упоминания, и критерии, проверяющие корректность информации?
7. Учтены ли исправления и финальные значения?
8. Не перепутаны ли разные тарифы, комиссии, лимиты и основания перевыпуска?
9. Каждая ли непустая цитата дословно присутствует в исходном диалоге?
10. Достаточно ли контекста в каждой цитате?
11. Не добавлены ли лишние цитаты?
12. Сохранены ли исходные названия и порядок критериев?
13. Содержит ли каждый объект только `name` и `mention`?
14. Является ли результат валидным JSON?

**Главное правило: не выдавай положительную оценку без прямого подтверждения словами сотрудника. Если подтверждающей цитаты нет или её недостаточно для однозначного вывода, возвращай пустую строку в `mention`.**
""".strip()

In [52]:
unique_names = sorted({
    item['name'].strip()
    for row in df['result'].dropna()
    for item in row
    if isinstance(item, dict) and item.get('name')
})

print(f"Количество уникальных name: {len(unique_names)}")


"Сотрудник озвучил корректно финальную стоимость перевыпуска карты/Сберкарты при плановом перевыпуске" in unique_names

Количество уникальных name: 32


True